In [1]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import pandas as pd 
import random
import json

from pathlib import Path

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg(folder):
    cfg = Config.fromfile(f'{folder}/vfnet_r18.py')
    return cfg


def get_best_configs():
    df1 = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/table_Single.pkl')
    df2 = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/table_Multi.pkl')
    
    return {'single':df1, 'multi':df2}


setup_cache_size_limit_of_dynamo()

single, multi = get_best_configs()['single'], get_best_configs()['multi']


In [2]:
def findWeight(folder):
    pth = os.listdir(folder)
    pth = [x for x in pth if x.endswith('.pth')]
    pth = [x for x in pth if x.startswith('epoch')]
    assert len(pth) == 1, f'Found {len(pth)} weights in {folder}'
    return f'{folder}/{pth[0]}'

In [3]:
def set_NoiseTesting_config(band_sel, single, multi, severity=0, corruption='gaussian'):
    """
    Set the configuration for noise testing.
    Args:
        band_sel (int): The selected band index.
        single (DataFrame): The single band information.
        multi (DataFrame): The multi band information.
        severity (int, optional): The severity of the corruption. Defaults to 0.
        corruption (str, optional): The type of corruption. Defaults to 'gaussian'.
    Returns:
        Config: The modified configuration object.
    """
    

    CORRUPTION = corruption
    SEVERITY = severity
    Seed = 42
    IMG_SIZE = 2048 # Default
    
    
    # Init Config
    _, LR, BS, ME = single.index[band_sel-1]
    base = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS'
    cfgDir = f'{base}/Single/perfect_b{band_sel}/{Seed}_BS_{BS}_LR_{LR}_ME_{ME}_OPT_SGD'
    cfg = init_cfg(folder=f'{cfgDir}')
    

    # Modify Config
    CHECKPOINT = findWeight(f'{cfgDir}')
    cfg.load_from = CHECKPOINT
    cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [1]},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(type='ImageCorruption', corruption=CORRUPTION, severity=SEVERITY), # 'gaussian', 'salt', 'pepper', 's&p' (salt and pepper), 'speckle', 'poisson'
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ]

    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    cfg.work_dir = work_dir
    
    return cfg


In [ ]:
# cfg = set_NoiseTesting_config(band_sel=1, single=single, multi=multi, severity=4, corruption='gaussian')

# # Run the test
# runner = RUNNERS.build(cfg)
# work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
# runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
# output_test_data = runner.test()

# Cycle Loop

In [4]:
Results = {'Band':[], 'Severity':[], 'AP':[], 'AP50':[], 'AP75':[]}

for band_sel in range(1, 13, 1):
    for S in np.arange(0, 5.2, 0.1):
        cfg = set_NoiseTesting_config(band_sel=band_sel, single=single, multi=multi, severity=S, corruption='gaussian')

        # Run the test
        runner = RUNNERS.build(cfg)
        work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
        runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
        output_test_data = runner.test()
        
        Results['Band'].append(band_sel)
        Results['Severity'].append(S)
        Results['AP'].append(output_test_data['coco/bbox_mAP'])
        Results['AP50'].append(output_test_data['coco/bbox_mAP_50'])
        Results['AP75'].append(output_test_data['coco/bbox_mAP_75'])

08/15 07:16:36 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/15 07:16:52 - mmengine - INFO - Epoch(test) [ 2/57]    eta: 0:03:45  time: 4.1012  data_time: 0.3365  memory: 1112  
08/15 07:16:52 - mmengine - INFO - Epoch(test) [ 3/57]    eta: 0:02:33  time: 2.8335  data_time: 0.2451  memory: 1112  
08/15 07:16:52 - mmengine - INFO - Epoch(test) [ 4/57]    eta: 0:01:57  time: 2.2155  data_time: 0.2008  memory: 1112  
08/15 07:16:52 - mmengine - INFO - Epoch(test) [ 5/57]    eta: 0:01:35  time: 1.8276  data_time: 0.1724  memory: 1112  
08/15 07:16:53 - mmengine - INFO - Epoch(test) [ 6/57]    eta: 0:01:19  time: 1.5652  data_time: 0.1524  memory: 1112  
08/15 07:16:53 - mmengine - INFO - Epoch(test) [ 7/57]    eta: 0:01:09  time: 1.3813  data_time: 0.1412  memory: 1112  
08/15 07:16:53 - mmengine - INFO - Epoch(test) [ 8/57]    eta: 0:01:01  time: 1.2525  data_time: 0.1288  memory: 1112  
08/15 07:16:54 - mmengine - INFO - Epoch(test) [ 9/57]    eta: 0:00:54  time: 1.1421  data_time: 0.1186  memory: 1112  
08/15 07:16:54 - mmengine - INFO - Epoch

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/15 07:17:18 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/15 07:17:18 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

...

08/15 08:12:29 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/15 08:12:29 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/15 08:13:12 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/15 08:13:12 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


08/15 08:13:57 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
08/15 08:13:57 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

In [ ]:
pd.DataFrame(Results).to_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/NoiseTesting.pkl')

# Plot

In [ ]:
b = get_metrics_x_band(df, 10)

b

In [ ]:
from style import set_style
from matplotlib import pyplot as plt

set_style()


df = pd.DataFrame(Results)


def get_metrics_x_band(df, band):
    df = df[df['Band'] == band]
    return df




b1 = get_metrics_x_band(df, 1)


fig = plt.figure()
for band in range(1, 13, 1):
    
    plt.subplot(2,6,band)

    b = get_metrics_x_band(df, band)
    x = b['Severity'].to_numpy()
    y = b['AP50'].to_numpy()
    
    plt.plot(b['Severity'], b['AP50'], label=f'Band {band}')
    plt.legend()

plt.show()
